[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Laboratorio: primer agente con `create_agent`.

**py501 · Aplicaciones Agénticas con LangChain** · Pythonista® · Instructor: José Luis Chiquete · Módulo 2: *El ecosistema LangChain v1*

> 📝 **Versión del alumno.** Las celdas marcadas con `# TU CÓDIGO AQUÍ` y las de *Tu respuesta* son para ti.

En este *notebook* se construirá el primer agente del asistente de soporte con `create_agent`: herramientas propias, *prompt* de sistema, salida estructurada, *streaming* y trazas en *LangSmith*.

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* **Ejercicio básico:** lo que todos deben dominar.
* **Reto (opcional):** para quien quiere profundizar. Se puede resolver como práctica.
* Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.
* Cada ejecución llama a un modelo de lenguaje: las respuestas varían entre ejecuciones y consumen tokens de tu clave.

### Preparación del entorno.

* La siguiente celda localizará `data/`, cargará `.env` si existe (ejecución local), definirá el modelo del curso y verificará la clave de API.

In [ ]:
import os

from dotenv import load_dotenv

RAIZ = next(p for p in (os.curdir, os.pardir)
            if os.path.isdir(os.path.join(p, "data", "politicas")))
load_dotenv(os.path.join(RAIZ, ".env"))   # solo tiene efecto en ejecución local
DATA = os.path.join(RAIZ, "data")
POLITICAS = os.path.join(DATA, "politicas")
MODELO = os.environ.get("MODELO", "google_genai:gemini-3.5-flash")
EMBEDDINGS = os.environ.get("EMBEDDINGS", "google_genai:gemini-embedding-001")

if not os.environ.get("GOOGLE_API_KEY") and MODELO.startswith("google_genai"):
    raise RuntimeError("Falta GOOGLE_API_KEY: agrégala como secreto de Codespaces y reconstruye el Codespace.")
MODELO

* La siguiente celda importará las bibliotecas de este *notebook*.

In [ ]:
from typing import Literal

from langchain.agents import create_agent
from pydantic import BaseModel

---
## 1. Herramientas.

Una herramienta es una función con **tipos** y **docstring**. El modelo solo ve el nombre, la descripción y el esquema de argumentos: escribe pensando en el modelo, no en otros programadores.

* La siguiente celda definirá las herramientas de políticas (las mismas del *notebook* `02`).

In [ ]:
from langchain.tools import tool


@tool
def listar_politicas() -> list[str]:
    """Lista los nombres de las políticas internas disponibles."""
    return sorted(f[:-3] for f in os.listdir(POLITICAS) if f.endswith(".md"))


@tool
def leer_politica(nombre: str) -> str:
    """Devuelve el texto completo de una política interna.

    Args:
        nombre: nombre exacto devuelto por listar_politicas, sin extensión.
    """
    ruta = os.path.join(POLITICAS, f"{nombre}.md")
    if not os.path.isfile(ruta):
        return f"No existe la política '{nombre}'. Disponibles: {', '.join(listar_politicas.invoke({}))}."
    with open(ruta, encoding="utf-8") as archivo:
        return archivo.read()

* La siguiente celda mostrará exactamente lo que el modelo ve de `leer_politica`.

In [ ]:
leer_politica.name, leer_politica.description, leer_politica.args

### Ejercicio 1 · La herramienta `crear_ticket`.

Escribe la herramienta `crear_ticket(titulo, descripcion, prioridad)` que agregue un diccionario a la lista `TICKETS` y devuelva el número del ticket.
* `prioridad` acepta solo `"baja"`, `"normal"` o `"urgente"` (usa `Literal`).
* El *docstring* debe explicar **cuándo** usar `"urgente"`.

In [ ]:
# TU CÓDIGO AQUÍ

---
## 2. El agente.

* La siguiente celda creará el agente con las herramientas disponibles (incluye `crear_ticket` si resolviste el ejercicio 1).

In [ ]:
SISTEMA = ("Eres el asistente de soporte interno de Andina Logística. "
           "Responde solo con base en las políticas; si no sabes, dilo. "
           "Crea un ticket cuando el usuario reporte un problema.")

herramientas = [listar_politicas, leer_politica]
if "crear_ticket" in globals():
    herramientas.append(crear_ticket)

agente = create_agent(model=MODELO, tools=herramientas, system_prompt=SISTEMA)
[h.name for h in herramientas]

* La siguiente celda mostrará el grafo de *LangGraph* que generó `create_agent`.

In [ ]:
print(agente.get_graph().draw_mermaid())

**Observa:** dos nodos, `model` y `tools`, con un ciclo entre ellos: es el mismo ciclo que se construyó a mano en el *notebook* `02`.

* La siguiente celda ejecutará el agente y mostrará todos los mensajes del ciclo.

In [ ]:
resultado = agente.invoke({"messages": [
    {"role": "user", "content": "¿Cuántos días de vacaciones tengo en mi tercer año?"}]})
for mensaje in resultado["messages"]:
    mensaje.pretty_print()

* La siguiente celda probará el flujo de tickets.

In [ ]:
resultado = agente.invoke({"messages": [
    {"role": "user", "content": "Se me cayó la laptop y ya no enciende, no puedo trabajar."}]})
resultado["messages"][-1].text, globals().get("TICKETS")

**💡 Ideas clave de la sección 2:**
* `create_agent` compila a un grafo `model` ↔ `tools`.
* El resultado es el estado final: la lista completa de mensajes.
* Cada ejecución queda trazada en *LangSmith*, proyecto **py501**.

---
## 3. Salida estructurada y *streaming*.

* La siguiente celda pedirá al agente una respuesta estructurada en `structured_response`.

In [ ]:
class RespuestaSoporte(BaseModel):
    respuesta: str
    politica_consultada: str | None
    ticket_creado: int | None


agente_estructurado = create_agent(MODELO, tools=herramientas, system_prompt=SISTEMA,
                                   response_format=RespuestaSoporte)
r = agente_estructurado.invoke({"messages": [{"role": "user", "content": "¿Cuál es el tope de hotel nacional?"}]})
r["structured_response"]

* La siguiente celda mostrará los pasos del agente a medida que ocurren con `stream_mode="updates"`.

In [ ]:
for evento in agente.stream({"messages": [{"role": "user", "content": "¿Cada cuánto renuevan las laptops?"}]},
                           stream_mode="updates"):
    for nodo, cambio in evento.items():
        print(f"[{nodo}]", cambio["messages"][-1].text or cambio["messages"][-1].tool_calls)

### Ejercicio 2 · El *docstring* importa.

Crea una versión de `leer_politica` cuyo *docstring* diga solo `"Lee un archivo."` y úsala en un agente nuevo con la pregunta de vacaciones. Revisa la traza: ¿qué cambió en el comportamiento?

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### Reto 1 · Campo `requiere_humano`.

Agrega el campo `requiere_humano: bool` a `RespuestaSoporte` y pruébalo con tres solicitudes: una duda simple, un robo de equipo y una queja laboral.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

---
## ✅ Cierre.

**Práctica:** terminar los ejercicios 1 y 2 y revisar las trazas en *LangSmith*.

**Siguiente *notebook*:** `05_herramientas_y_contexto.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>